# Reading answer scripts with a VLMProduces one Markdown file per input image, named identically, ready toscore with `modules/06_evaluation/src/ocr_bench.py`.**Before uploading:** cover pages (`page_01`) carry names, USNs andmarks. They are already excluded by `07_reconstruct`, but check your zip.**Runtime → Change runtime type → T4 GPU** before running anything.

## 1. Confirm the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Install

In [ ]:
!pip -q install "transformers>=4.49" accelerate qwen-vl-utils bitsandbytes

## 3. Upload your pagesZip the images first. Filenames become the output names, so use thepage ids the benchmark expects, e.g. `s06_c1_p05.png`.

In [ ]:
import zipfile, pathlibfrom google.colab import filesup = files.upload()                 # choose your .zipname = next(iter(up))IN = pathlib.Path('/content/pages'); IN.mkdir(exist_ok=True)with zipfile.ZipFile(name) as z:    z.extractall(IN)imgs = sorted(p for p in IN.rglob('*') if p.suffix.lower() in {'.png','.jpg','.jpeg'})print(f'{len(imgs)} image(s)')for p in imgs[:10]: print('  ', p.name)

## 4. Load the model`3B` is comfortable on a T4 and fast. `7B` is more accurate but needs4-bit to fit — try 3B first and only move up if the score demands it.**The pixel budget is set on the PROCESSOR here, and that is the partthat matters.** Qwen's default cap is 16384 image patches. A1598x2177 scan is 3.48M pixels, which at 28x28 patches is ~4,400visual tokens, and attention over 4,400 tokens is what asks a T4 for18.85 GiB and dies. Capping at 1024 patches resizes the page to ~800kpixels first, which is ~19x less attention memory and still well abovewhat this handwriting needs to stay legible.Setting it on the processor makes it apply no matter how the image ispassed in later. Putting the same numbers only inside the chat messagedoes NOT: the message is a template, and a raw PIL image handedstraight to `processor(images=...)` sails past it.

In [ ]:
# --- pick one, run the whole notebook, then come back and pick the# --- other. ENGINE names the output folder, so the two runs land in# --- separate directories and ocr_bench can score them side by side.RUN = "7b"          # "3b" or "7b"if RUN == "3b":    MODEL, FOUR_BIT, ENGINE = "Qwen/Qwen2.5-VL-3B-Instruct", False, "qwen3b_v2"else:    MODEL, FOUR_BIT, ENGINE = "Qwen/Qwen2.5-VL-7B-Instruct", True, "qwen7b"# in 28x28 patches. Lower MAX_PATCHES if you hit OOM.MIN_PATCHES, MAX_PATCHES = 256, 1024import osos.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"import torchfrom transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessorkw = dict(torch_dtype=torch.bfloat16, device_map="auto")if FOUR_BIT:    from transformers import BitsAndBytesConfig    kw["quantization_config"] = BitsAndBytesConfig(        load_in_4bit=True,        bnb_4bit_compute_dtype=torch.bfloat16,        bnb_4bit_quant_type="nf4",        bnb_4bit_use_double_quant=True,    )model = Qwen2_5_VLForConditionalGeneration.from_pretrained(MODEL, **kw)model.eval()processor = AutoProcessor.from_pretrained(    MODEL,    min_pixels=MIN_PATCHES * 28 * 28,    max_pixels=MAX_PATCHES * 28 * 28,)print('engine :', ENGINE)print('model  :', MODEL, '(4-bit)' if FOUR_BIT else '(bf16)')print('max visual tokens per image:', MAX_PATCHES)

## 5. The prompt

In [ ]:
PROMPT = """You are transcribing a handwritten exam answer that ahuman will mark. Your only job is to report what is on the paper.THE ONE RULE THAT MATTERSYou are not answering this exam and you are not helping the student.Do not use what you know about the subject to fill in, complete orcorrect anything. If the page shows a worked example you recognise,that recognition is a trap: transcribe the marks that are there, evenwhere they contradict what the answer should be. A wrong value copiedfaithfully is correct output. A right value you supplied is a seriouserror, because the marker cannot tell you invented it.WHEN YOU CANNOT READ SOMETHINGWrite [?] in place of the word or number. Do this readily - an answerpeppered with [?] is far more useful than a fluent one that is partlyinvented. Never substitute a plausible word for an illegible one.TABLESTranscribe tables as Markdown tables. Most are comparisons - twocolumns of words - and you should read them normally.The care is needed per CELL, not per table. Any single cell you cannotread with certainty becomes [?]. Never infer a cell's value from thepattern of the cells around it: a column of numbers that looks like itcontinues a sequence is exactly where a wrong value gets invented.Only if MOST of the cells would be [?] - a dense numeric working youcannot resolve - skip the table entirely and emit![table](x1,y1,x2,y2)using the box rule below.DIAGRAMSDiagrams, graphs, figures, flowcharts, timing charts, circuitsketches: never describe them in prose and never transcribe the labelsinside them as text. Emit a placeholder with the box that encloses thewhole figure, including its labels:![diagram](x1,y1,x2,y2)Coordinates are pixels in the image as you see it: x1,y1 is thetop-left corner and x2,y2 the bottom-right. Give one box per distinctfigure. Make the box tight around the drawing but do not clip any partof it, and do not merge two separate figures into one box.STRUCTURE- Question numbers appear in the left margin (1, 2a, 2b, 2c, 3a, 3b,  4a, 4b). Emit each as: ### 2a)- Sub-parts (i, ii, iii ... or a, b, c ...) as: #### i)- Mathematics: inline LaTeX between $ ... $- Struck-out or cancelled text: wrap in ~~ ~~- Keep the line breaks as written.- Preserve the student's spelling, grammar and arithmetic exactly,  errors included.Output only the Markdown. No commentary, no preamble."""print(PROMPT)

## 6. Read every page`process_vision_info` is what actually resizes the image to theprocessor's pixel budget. Passing a raw PIL image to`processor(images=...)` skips that step, which is how a page becomes~4,400 visual tokens and asks for 18.85 GiB.**Boxes come back in the RESIZED image's coordinates, not the page's.**At 1024 patches a 1598x2177 page is seen at roughly 768x1046, so a boxthe model gives as `(120,340,980,760)` means nothing against theoriginal file until it is scaled by the ratio between the two. Thatrescale happens here, and the boxes written to the Markdown are inORIGINAL page pixels - directly usable for cropping the figure out.A page that still OOMs is retried once at half the patch budget ratherthan being lost, and the message says so - a page silently written asan empty file would score as a total miss and look like a readingfailure rather than a memory one.

In [ ]:
import time, pathlib, gc, refrom PIL import Imagefrom qwen_vl_utils import process_vision_infoOUT = pathlib.Path('/content') / ENGINE; OUT.mkdir(exist_ok=True)BOX = re.compile(r'!\[(diagram|table)\]\(\s*([\d\.]+)\s*,\s*([\d\.]+)\s*,'                 r'\s*([\d\.]+)\s*,\s*([\d\.]+)\s*\)')def rescale_boxes(body, seen_size, true_size):    """Model boxes are in the resized image; put them back on the page."""    (sw, sh), (tw, th) = seen_size, true_size    if not sw or not sh:        return body    fx, fy = tw / sw, th / sh    def fix(m):        kind = m.group(1)        x1, y1, x2, y2 = (float(m.group(i)) for i in range(2, 6))        # clamp, because a box that runs off the edge crops to nothing        x1, x2 = sorted((max(0, x1 * fx), min(tw, x2 * fx)))        y1, y2 = sorted((max(0, y1 * fy), min(th, y2 * fy)))        return f'![{kind}]({int(x1)},{int(y1)},{int(x2)},{int(y2)})'    return BOX.sub(fix, body)def read(path, max_patches=MAX_PATCHES):    image = Image.open(path).convert('RGB')    true_size = image.size    messages = [{"role": "user", "content": [        {"type": "image", "image": image,         "min_pixels": MIN_PATCHES * 28 * 28,         "max_pixels": max_patches * 28 * 28},        {"type": "text", "text": PROMPT}]}]    text = processor.apply_chat_template(        messages, tokenize=False, add_generation_prompt=True)    # THIS is the resize step. Without it the pixel budget is ignored.    image_inputs, _ = process_vision_info(messages)    # what the model actually saw - the basis for every box it returns    seen_size = image_inputs[0].size    inputs = processor(text=[text], images=image_inputs,                       padding=True, return_tensors="pt").to(model.device)    tokens = int(inputs.input_ids.shape[1])    with torch.no_grad():        out = model.generate(**inputs, max_new_tokens=1536, do_sample=False)    trimmed = out[0][tokens:]    body = processor.decode(trimmed, skip_special_tokens=True).strip()    body = rescale_boxes(body, seen_size, true_size)    del inputs, out    return body, tokenst0 = time.time()failed = []for i, p in enumerate(imgs, 1):    started = time.time()    body, tokens = '', 0    for attempt, budget in enumerate([MAX_PATCHES, MAX_PATCHES // 2]):        try:            body, tokens = read(p, budget)            if attempt:                print(f'    (recovered at {budget} patches)')            break        except torch.cuda.OutOfMemoryError:            torch.cuda.empty_cache(); gc.collect()            if attempt:                failed.append(p.stem)                print(f'  {p.stem}: OOM even at {budget} patches')        except Exception as e:            failed.append(p.stem)            print(f'  {p.stem}: FAILED {type(e).__name__}: {e}')            break    (OUT / (p.stem + '.md')).write_text(body, encoding='utf-8')    print(f'[{i}/{len(imgs)}] {p.stem}  {len(body)} chars  '          f'{tokens} in-tokens  {time.time()-started:.0f}s', flush=True)    torch.cuda.empty_cache(); gc.collect()print(f'\nTotal {time.time()-t0:.0f}s')if failed:    print(f'FAILED ({len(failed)}): {failed}')else:    print('all pages read')

## 7. Did the prompt actually take?The previous run scored 0.141 CER overall but fabricated a wholeDijkstra table on the one page that had one, and used `[?]` exactlyzero times across every page - including that one. So the useful checkis not the score, it is whether the model is now willing to decline.`[?]` and `![table]` counts of zero mean the new instructions didnothing, whatever the CER says.

In [ ]:
import re, collectionsmarks = collections.Counter()unboxed = 0for f in sorted(OUT.glob('*.md')):    body = f.read_text(encoding='utf-8')    marks['[?]'] += len(re.findall(r'\[\?\]', body))    marks['![diagram] boxed'] += len([b for b in BOX.findall(body)                                      if b[0] == 'diagram'])    marks['![table] boxed'] += len([b for b in BOX.findall(body)                                    if b[0] == 'table'])    marks['md table rows'] += len([l for l in body.splitlines()                                   if l.strip().startswith('|')])    # a placeholder with no coordinates is still useful text but is no    # use for cropping, so it is counted separately rather than hidden    unboxed += len(re.findall(r'!\[(?:diagram|table)\](?!\s*\()', body))marks['placeholders WITHOUT a box'] = unboxedprint(f'{"signal":<28}{"count":>7}')for k in ['[?]', '![diagram] boxed', '![table] boxed',          'placeholders WITHOUT a box', 'md table rows']:    print(f'{k:<28}{marks[k]:>7}')print()if marks['[?]'] == 0:    print('WARNING: still never declines. The anti-fabrication clause '          'is not working.')else:    print('Good: it is declining where it cannot read.')if unboxed:    print(f'WARNING: {unboxed} placeholder(s) came back with no '          f'coordinates, so nothing can be cropped for them.')if marks['md table rows'] and not marks['[?]']:    print('NOTE: tables were transcribed with no [?] anywhere. Check '          'those cells against the page before trusting them - this is '          'exactly where the 3B invented values last time.')

## 8. Draw the boxes back onto the pagesA wrong box is invisible in the Markdown and obvious in a picture. Ifthese do not sit tightly around the figures, the coordinates are notusable for cropping and nothing downstream will work.

In [ ]:
from PIL import ImageDrawshown = 0for f in sorted(OUT.glob('*.md')):    body = f.read_text(encoding='utf-8')    boxes = BOX.findall(body)    if not boxes:        continue    src = [p for p in imgs if p.stem == f.stem]    if not src:        continue    im = Image.open(src[0]).convert('RGB')    draw = ImageDraw.Draw(im)    for kind, x1, y1, x2, y2 in boxes:        colour = (200, 30, 30) if kind == 'diagram' else (30, 90, 200)        draw.rectangle([float(x1), float(y1), float(x2), float(y2)],                       outline=colour, width=6)        draw.text((float(x1) + 8, float(y1) + 8), kind, fill=colour)    im.thumbnail((700, 700))    print(f'{f.stem}: {len(boxes)} box(es)')    display(im)    shown += 1    if shown >= 4:        breakif not shown:    print('No boxed placeholders were produced at all - either these '          'pages have no figures, or the box instruction did not take.')

## 9. Spot-check the page that failed last time

In [ ]:
target = 's10_c2_p10'hit = [f for f in OUT.glob('*.md') if f.stem == target]print((hit[0] if hit else sorted(OUT.glob('*.md'))[0]).read_text(encoding='utf-8')[:2000])

## 9. DownloadUnzip into `modules/06_evaluation/predictions/<ENGINE>/`, then locally:```python modules/06_evaluation/src/ocr_bench.py --engine qwen7b --verbosepython modules/06_evaluation/src/ocr_bench.py --engine qwen3b --verbose   # the old run```Same pages, same scorer, so the three engines line up directly againstthe 0.573 baseline.

In [ ]:
import shutilfrom google.colab import filesshutil.make_archive(f'/content/{ENGINE}', 'zip', OUT)files.download(f'/content/{ENGINE}.zip')

## What to look for- **`[?]` markers** — the model admitting it cannot read. Zero of these  across a whole booklet is a red flag, not a good sign.- **`![table]`** on the Dijkstra page. Last run it invented  `5 | 6 | 7 | 8` where the page says `2,A | 5,A | inf | inf`.- **`![diagram]`** where a figure is, rather than prose describing it.- **Question headings** (`### 2a)`) — the assembly step groups on these,  so they matter more than the prose around them.- **A LOWER CER is not automatically better.** If the model starts  emitting `![table]` where it used to invent one, CER may barely move  while the output becomes far more trustworthy. Read the diff, not  just the number.